# Hamilton STAR

The Hamilton STAR(let) is a liquid handler with independent pipetting channels, an optional 96-head, and an optional iSWAP plate transport arm.

In this notebook, you will learn how to use PyLabRobot to set up the STAR, pick up tips, and move liquid between wells.

**Prerequisites:**

- Installed PyLabRobot with USB support: `pip install pylabrobot[usb]`
- Platform-specific driver setup (libusb on Mac, Zadig on Windows) — see [the installation guide](../../_getting-started/installation)
- Connected the Hamilton to your computer using the USB cable

Video of what this code does:

<iframe width="640" height="360" src="https://www.youtube.com/embed/NN6ltrRj3bU" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and one of the Hamilton deck classes. `STAR` is the device that owns the driver and exposes capabilities like `pip` (independent channels), `head96` (96-head), and `iswap` (plate transport arm).

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARLetDeck

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

After `setup()`, the driver discovers installed hardware automatically. `star.pip` is always available. `star.head96` and `star.iswap` are `None` if the corresponding hardware is not installed.

## Creating the deck layout

Define the physical deck layout by assigning carriers with tip racks and plates. This tutorial uses:

- {class}`~pylabrobot.resources.hamilton.tip_carriers.TIP_CAR_480_A00` tip carrier
- {class}`~pylabrobot.resources.hamilton.plate_carriers.PLT_CAR_L5AC_A00` plate carrier
- {class}`~pylabrobot.resources.corning_costar.plates.Cor_96_wellplate_360ul_Fb` 96-well plate
- {class}`~pylabrobot.resources.hamilton.tip_racks.hamilton_96_tiprack_1000uL_filter` tip rack

In [ ]:
from pylabrobot.resources import (
  TIP_CAR_480_A00,
  PLT_CAR_L5AC_A00,
  Cor_96_wellplate_360ul_Fb,
  hamilton_96_tiprack_1000uL_filter,
)

tip_car = TIP_CAR_480_A00(name="tip carrier")
tip_car[0] = hamilton_96_tiprack_1000uL_filter(name="tips_01")
deck.assign_child_resource(tip_car, rails=3)

plt_car = PLT_CAR_L5AC_A00(name="plate carrier")
plt_car[0] = Cor_96_wellplate_360ul_Fb(name="plate_01")
deck.assign_child_resource(plt_car, rails=15)

Let's look at a summary of the deck layout using {meth}`~pylabrobot.resources.deck.Deck.summary`.

In [ ]:
deck.summary()

## Picking up tips

Use `star.pip` — the independent-channel pipetting capability — to pick up tips. Indexing a tip rack with `["A1:C1"]` returns the first three tip spots in column 1.

In [ ]:
tiprack = deck.get_resource("tips_01")
await star.pip.pick_up_tips(tiprack["A1:C1"])

## Aspirating and dispensing

Aspirate from wells `A1:C1` and dispense to `D1:F1`, using channels 0, 1, and 2.

In [ ]:
plate = deck.get_resource("plate_01")
await star.pip.aspirate(plate["A1:C1"], vols=[100.0, 50.0, 200.0])

In [ ]:
await star.pip.dispense(plate["D1:F1"], vols=[100.0, 50.0, 200.0])

Move the liquid back:

In [ ]:
await star.pip.aspirate(plate["D1:F1"], vols=[100.0, 50.0, 200.0])
await star.pip.dispense(plate["A1:C1"], vols=[100.0, 50.0, 200.0])

## Dropping tips

Return tips to their original positions:

In [ ]:
await star.pip.drop_tips(tiprack["A1:C1"])

## Backend parameters

For STAR-specific tuning, pass `backend_params` to any operation. The available parameter classes are:

- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.PickUpTipsParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.DropTipsParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.AspirateParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.DispenseParams`

For example, to use liquid level detection during aspiration:

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.pip_backend import STARPIPBackend, LLDMode

await star.pip.pick_up_tips(tiprack["D1:F1"])

await star.pip.aspirate(
  plate["A1:C1"],
  vols=[100.0, 100.0, 100.0],
  backend_params=STARPIPBackend.AspirateParams(
    lld_mode=[LLDMode.GAMMA, LLDMode.GAMMA, LLDMode.GAMMA],
  ),
)

await star.pip.dispense(plate["D1:F1"], vols=[100.0, 100.0, 100.0])
await star.pip.drop_tips(tiprack["D1:F1"])

## Teardown

In [ ]:
await star.stop()